In [1]:
!pip install -q -U transformers datasets accelerate evaluate jiwer
!pip install -q -U soundfile librosa mutagen
!pip install -q -U datacollective huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 92.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.7/195.7 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 kB 19.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 99.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━

In [2]:
import os
import csv
import glob
import gc
import shutil
import random
import warnings

import numpy as np
import pandas as pd
import torch
import soundfile as sf
from mutagen.mp3 import MP3

from datasets import Dataset, Audio, load_dataset, concatenate_datasets, load_from_disk
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
import evaluate
from huggingface_hub import login, hf_hub_download, upload_file

warnings.filterwarnings("ignore")


REPO_ID = "amirsz8203/whisper-small-fa-finetuned"   
MODEL_NAME = REPO_ID                                
LANGUAGE = "persian"
TASK = "transcribe"
SAMPLE_RATE = 16000
SEED = 42

ROUND = 2
TARGET_CV_HOURS = 15         
REPLAY_RATIO = 0.15           

CV_EXTRACT_DIR = "/tmp/common_voice_fa_extracted"
OUTPUT_DIR = f"/kaggle/working/whisper-small-fa-finetuned-round{ROUND}"
PROCESSED_DIR = "/kaggle/working/processed_dataset"
USED_CLIPS_PATH = "/kaggle/working/used_clips.txt"

random.seed(SEED)
np.random.seed(SEED)

n_gpu = torch.cuda.device_count()
print("تعداد GPU شناسایی‌شده:", n_gpu)
for i in range(n_gpu):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = "خودتان جایگذاری کنید"  

login(token=HF_TOKEN)

تعداد GPU شناسایی‌شده: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4


In [3]:
try:
    MDC_API_KEY = UserSecretsClient().get_secret("MDC_API_KEY")
except Exception:
    MDC_API_KEY = "خودتان جایگذاری کنید"

os.environ["MDC_API_KEY"] = MDC_API_KEY

CV_DATASET_ID = "cmqinhw5100v8nr07gyg5gi4v"  # Common Voice Scripted Speech - Persian

from datacollective import download_dataset
import tarfile

cv_archive_path = str(download_dataset(CV_DATASET_ID))
print("مسیر آرشیو دانلود‌شده:", cv_archive_path)

if os.path.isfile(cv_archive_path):
    if not os.path.isdir(CV_EXTRACT_DIR) or not os.listdir(CV_EXTRACT_DIR):
        os.makedirs(CV_EXTRACT_DIR, exist_ok=True)
        print("در حال extract کردن آرشیو... (ممکنه چند دقیقه طول بکشه)")
        with tarfile.open(cv_archive_path, "r:gz") as tar:
            tar.extractall(path=CV_EXTRACT_DIR)
        print("extract تموم شد.")
    else:
        print("قبلاً extract شده، از کش استفاده می‌کنیم.")
    cv_root = CV_EXTRACT_DIR
elif os.path.isdir(cv_archive_path):
    cv_root = cv_archive_path
else:
    raise RuntimeError(f"مسیر برگشتی نه فایله نه پوشه: {cv_archive_path}")

print("مسیر نهایی برای جستجوی tsv/mp3:", cv_root)

█████████████████████████████████████████████████🦊 100.0% (10.5 GB/10.5 GB) Average: 68.9 MB/s Total time: 02:35
مسیر آرشیو دانلود‌شده: /root/.mozdata/datasets/common-voice-scripted-speech-26-0-persia-65a9441e.tar.gz
در حال extract کردن آرشیو... (ممکنه چند دقیقه طول بکشه)
extract تموم شد.
مسیر نهایی برای جستجوی tsv/mp3: /tmp/common_voice_fa_extracted


In [4]:
tsv_candidates = (
    glob.glob(os.path.join(cv_root, "**", "validated.tsv"), recursive=True)
    or glob.glob(os.path.join(cv_root, "**", "train.tsv"), recursive=True)
    or glob.glob(os.path.join(cv_root, "**", "*.tsv"), recursive=True)
)
assert tsv_candidates, "هیچ فایل .tsv پیدا نشد."
cv_tsv_path = tsv_candidates[0]

mp3_candidates = glob.glob(os.path.join(cv_root, "**", "*.mp3"), recursive=True)
assert mp3_candidates, "هیچ فایل mp3 پیدا نشد."
clips_dir = os.path.dirname(mp3_candidates[0])

cv_df = pd.read_csv(cv_tsv_path, sep="\t", quoting=csv.QUOTE_NONE)
assert "path" in cv_df.columns and "sentence" in cv_df.columns

cv_df["full_path"] = cv_df["path"].apply(lambda p: os.path.join(clips_dir, p))
# همون شافل ثابت راند ۱ -> ترتیب کاندیدها بین راندها سازگار می‌مونه
cv_df = cv_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)


def mp3_duration_sec(path):
    try:
        return MP3(path).info.length
    except Exception:
        return None


print("تعداد کل ردیف‌های متادیتا:", len(cv_df))

تعداد کل ردیف‌های متادیتا: 341657


In [5]:
# ---------------- کلیپ‌های استفاده‌شده در راندهای قبل + انتخاب replay ----------------
try:
    used_file = hf_hub_download(repo_id=REPO_ID, filename="used_clips.txt")
    with open(used_file) as f:
        previous_clips = [line.strip() for line in f if line.strip()]
    print(f"{len(previous_clips)} کلیپ از راند(های) قبل پیدا شد.")
except Exception as e:
    previous_clips = []
    print("used_clips.txt روی هاب پیدا نشد (یعنی این اولین رانده):", e)

random.seed(SEED)
n_replay = int(len(previous_clips) * REPLAY_RATIO)
replay_paths = set(random.sample(previous_clips, n_replay)) if n_replay > 0 else set()
used_paths_exclude = set(previous_clips) - replay_paths

print(f"replay این راند: {len(replay_paths)} کلیپ از راند(های) قبل")
print(f"exclude (غیر-replay): {len(used_paths_exclude)} کلیپ")

used_clips.txt:   0%|          | 0.00/1.74M [00:00<?, ?B/s]

18327 کلیپ از راند(های) قبل پیدا شد.
replay این راند: 2749 کلیپ از راند(های) قبل
exclude (غیر-replay): 15578 کلیپ


In [6]:
# ---------------- انتخاب ۱۵ ساعت جدید + کلیپ‌های replay ----------------
selected = []
total_sec = 0.0
target_sec = TARGET_CV_HOURS * 3600
replay_selected = []
replay_sec = 0.0

for _, row in cv_df.iterrows():
    path = row["full_path"]
    sentence = str(row["sentence"])
    if len(sentence) > 400:
        continue

    if path in replay_paths:
        dur = mp3_duration_sec(path)
        if dur is None:
            continue
        replay_selected.append({"path": path, "sentence": sentence, "source": "common_voice_replay"})
        replay_sec += dur
        continue

    if path in used_paths_exclude:
        continue

    dur = mp3_duration_sec(path)
    if dur is None:
        continue
    selected.append({"path": path, "sentence": sentence, "source": "common_voice"})
    total_sec += dur
    if total_sec >= target_sec:
        break

print(f"داده‌ی جدید این راند: {len(selected)} کلیپ، {total_sec/3600:.2f} ساعت")
print(f"replay: {len(replay_selected)} کلیپ، {replay_sec/3600:.2f} ساعت")

cv_records = selected + replay_selected
random.shuffle(cv_records)
print(f"مجموع این راند: {len(cv_records)} کلیپ، {(total_sec + replay_sec)/3600:.2f} ساعت")

# used_clips.txt جدید برای push در پایان: قبلی‌ها + این راند (بدون تکرار)
all_used_paths = sorted(set(previous_clips) | {r["path"] for r in selected})
with open(USED_CLIPS_PATH, "w") as f:
    for p in all_used_paths:
        f.write(p + "\n")
print("تعداد کل کلیپ‌های استفاده‌شده تا الان (برای used_clips.txt):", len(all_used_paths))

داده‌ی جدید این راند: 13689 کلیپ، 15.00 ساعت
replay: 2749 کلیپ، 2.99 ساعت
مجموع این راند: 16438 کلیپ، 17.99 ساعت
تعداد کل کلیپ‌های استفاده‌شده تا الان (برای used_clips.txt): 32016


In [7]:
full_df = pd.DataFrame(cv_records)
print("مجموع کلیپ‌ها:", len(full_df))
print(full_df["source"].value_counts())

full_ds = Dataset.from_pandas(full_df.reset_index(drop=True))
full_ds = full_ds.rename_column("path", "audio")
full_ds = full_ds.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))

split_ds = full_ds.train_test_split(test_size=0.1, seed=SEED)
train_ds = split_ds["train"]
eval_ds = split_ds["test"]

print("Train:", len(train_ds), " | Eval:", len(eval_ds))

مجموع کلیپ‌ها: 16438
source
common_voice           13689
common_voice_replay     2749
Name: count, dtype: int64
Train: 14794  | Eval: 1644


In [8]:
# processor/tokenizer/feature_extractor رو از خروجی راند قبل لود می‌کنیم (نه از openai/whisper-small خام)
feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)
processor = WhisperProcessor.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)

processor_config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.13k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.93M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

In [9]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch


train_ds = train_ds.map(prepare_dataset, remove_columns=train_ds.column_names)
eval_ds = eval_ds.map(prepare_dataset, remove_columns=eval_ds.column_names)

train_ds = train_ds.filter(lambda b: len(b["labels"]) <= 225)
eval_ds = eval_ds.filter(lambda b: len(b["labels"]) <= 225)
print("بعد از فیلتر -> Train:", len(train_ds), " | Eval:", len(eval_ds))

Map:   0%|          | 0/14794 [00:00<?, ? examples/s]

Map:   0%|          | 0/1644 [00:00<?, ? examples/s]

Filter:   0%|          | 0/14794 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1644 [00:00<?, ? examples/s]

بعد از فیلتر -> Train: 14794  | Eval: 1644


In [10]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union


@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [11]:
metric = evaluate.load("wer")


def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

In [12]:
# مدل رو از خروجی راند ۱ لود می‌کنیم، نه از openai/whisper-small خام
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
model.generation_config.language = LANGUAGE
model.generation_config.task = TASK
model.generation_config.forced_decoder_ids = None

model.safetensors: reconstructing file:   0%|          |  0.00B /  967MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/4.59k [00:00<?, ?B/s]

In [13]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=4,
    gradient_checkpointing=True,
    learning_rate=5e-6,           # پایین‌تر از راند ۱ (1e-5) چون مدل دیگه از صفر شروع نمی‌کنه
    warmup_steps=50,
    num_train_epochs=2,           # کمتر از راند ۱ (3) برای جلوگیری از overfit/فراموشی روی داده‌ی جدید
    fp16=torch.cuda.is_available(),
    eval_strategy="steps",
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=200,
    eval_steps=200,
    logging_steps=25,
    save_total_limit=2,
    report_to=["none"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
)

print("n_gpu که Trainer می‌بینه:", trainer.args.n_gpu, "| batch مؤثر کل =",
      training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps * max(trainer.args.n_gpu, 1))

n_gpu که Trainer می‌بینه: 2 | batch مؤثر کل = 32


In [14]:
trainer.train()

Step,Training Loss,Validation Loss,Wer
200,1.058841,0.251815,30.995061
400,1.064779,0.241871,31.104811
600,0.699717,0.238103,29.376258
800,0.737131,0.234822,29.421986
926,0.704722,0.233237,28.507408


[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The cu

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


TrainOutput(global_step=926, training_loss=0.8713211681750114, metrics={'train_runtime': 16427.8705, 'train_samples_per_second': 1.801, 'train_steps_per_second': 0.056, 'total_flos': 8.53866482466816e+18, 'train_loss': 0.8713211681750114, 'epoch': 2.0})

In [15]:
save_dir = OUTPUT_DIR + "/final"
trainer.save_model(save_dir)
processor.save_pretrained(save_dir)
print("مدل ذخیره شد در:", save_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

مدل ذخیره شد در: /kaggle/working/whisper-small-fa-finetuned-round2/final


In [52]:
model.eval()
idx = random.randint(0, len(eval_ds) - 1)
sample = eval_ds[idx]

input_features = torch.tensor(sample["input_features"]).unsqueeze(0)
if torch.cuda.is_available():
    input_features = input_features.to(model.device)

with torch.no_grad():
    predicted_ids = model.generate(input_features)

predicted_text = processor.batch_decode(predicted_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

label_ids = [l if l != -100 else tokenizer.pad_token_id for l in sample["labels"]]
actual_text = processor.batch_decode([label_ids], skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

print("متن واقعی:      ", actual_text)
print("متن پیش‌بینی‌شده:", predicted_text)

متن واقعی:       !من مونیکا رو نمی بینم
متن پیش‌بینی‌شده: من مونیکا رو نمی بینم


In [26]:
HOURS_THIS_ROUND = TARGET_CV_HOURS

trainer.args.hub_model_id = REPO_ID
trainer.push_to_hub(
    commit_message=f"round {ROUND}: fine-tuned on +{HOURS_THIS_ROUND}h Persian Common Voice (+ {REPLAY_RATIO:.0%} replay from round 1)",
)
processor.push_to_hub(REPO_ID)

upload_file(
    path_or_fileobj=USED_CLIPS_PATH,
    path_in_repo="used_clips.txt",
    repo_id=REPO_ID,
    repo_type="model",
)
print("راند ۲ کامل push شد به:", REPO_ID)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


راند ۲ کامل push شد به: amirsz8203/whisper-small-fa-finetuned
